# 23 - Paper-aligned brain correlation

Companion notes: [`docs/guides/original_paper_methodology.md`](../docs/guides/original_paper_methodology.md) and [`docs/notes/paper_reproduction_and_ml_next_steps.md`](../docs/notes/paper_reproduction_and_ml_next_steps.md).

This notebook separates two analyses that are easy to conflate.

1. **Beta-only diagnostic:** directly compare repeat-averaged vision and imagery beta patterns in the exact Table 1 ROIs. This needs no reconstruction model, but it is *not* Table 1.
2. **Exact Table 1 metric:** reconstruct an image, pass that reconstruction through the pretrained GNet image-to-brain encoder, and correlate the GNet-predicted beta pattern with the measured beta pattern across voxels.

The paper's early ROI is V1 through hV4. Its higher ROI is `nsdgeneral` minus that early ROI. This differs from Notebook 20, which deliberately remains unchanged for result provenance.

In [6]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'nsdimagery').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'nsdimagery').is_dir():
    raise RuntimeError('Start Jupyter from the repository root or notebooks/.')
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from nsdimagery import (
    build_event_table, extract_masked_betas, find_data_root, load_roi,
    mask_at_coordinates, paper_visual_roi_masks, paths_for_subject,
    reconstruction_brain_correlations, zscore_within_groups,
)

DATA_ROOT = find_data_root(REPO_ROOT)
OUTPUT_DIR = REPO_ROOT / 'outputs' / '05_paper_brain_correlation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO_ROOT)
print('Data root:', DATA_ROOT)

Repository: /home/jovyan/NHprojectNSDimagery
Data root: /home/jovyan/NHprojectNSDimagery/data/nsd


## 1. Beta-only diagnostic in the paper ROIs

For each subject, set, and target, the code Z-scores every voxel separately within each run and averages the 8 vision or 16 imagery repeats. It then computes a spatial Pearson correlation across ROI voxels between the measured vision and imagery patterns. The target-specific contrast is the mean matched-target correlation minus the mean mismatched-target correlation.

A positive contrast says that the same target has more similar vision/imagery patterns than different targets. It does **not** measure reconstruction quality and should not be compared numerically with Table 1.

In [7]:
SUBJECTS = tuple(range(1, 9))
STIMULUS_SETS = ('A', 'B')
TABLE_REGIONS = ('early_visual', 'higher_visual', 'visual_cortex')

def load_paper_patterns(subject):
    general, _ = load_roi(DATA_ROOT, subject, 'nsdgeneral')
    prf, _ = load_roi(DATA_ROOT, subject, 'prf-visualrois')
    betas, coordinates = extract_masked_betas(
        paths_for_subject(DATA_ROOT, subject)['beta'], general
    )
    events = build_event_table(DATA_ROOT, subject).reset_index(drop=True)
    patterns = betas[events['beta_index'].to_numpy()]
    patterns = zscore_within_groups(patterns, events['run_name'].to_numpy())
    volume_masks = paper_visual_roi_masks(general, prf)
    column_masks = {
        name: mask_at_coordinates(mask, coordinates)
        for name, mask in volume_masks.items()
    }
    return events, patterns, column_masks

def condition_centroids(events, patterns, task, stimulus_set):
    rows = []
    for target in range(1, 7):
        selected = (
            events['task'].eq(task)
            & events['stimulus_set'].eq(stimulus_set)
            & events['target_number'].eq(target)
        ).to_numpy()
        rows.append(patterns[selected].mean(axis=0))
    return np.stack(rows)

In [8]:
rows = []
for subject in SUBJECTS:
    print(f'subj{subject:02d}')
    events, patterns, masks = load_paper_patterns(subject)
    for stimulus_set in STIMULUS_SETS:
        vision = condition_centroids(events, patterns, 'vision', stimulus_set)
        imagery = condition_centroids(events, patterns, 'imagery', stimulus_set)
        for region in TABLE_REGIONS:
            selected = masks[region]
            matrix = np.empty((6, 6), dtype=float)
            for vision_target in range(6):
                repeated = np.repeat(vision[vision_target][None, :], 6, axis=0)
                matrix[vision_target] = reconstruction_brain_correlations(
                    repeated, imagery, voxel_mask=selected
                )
            matched = np.diag(matrix)
            mismatched = matrix[~np.eye(6, dtype=bool)]
            rows.append({
                'subject': f'subj{subject:02d}',
                'stimulus_set': stimulus_set,
                'region': region,
                'matched_correlation': matched.mean(),
                'mismatched_correlation': mismatched.mean(),
                'target_specific_contrast': matched.mean() - mismatched.mean(),
                'n_voxels': int(selected.sum()),
            })

beta_only = pd.DataFrame(rows)
beta_only.to_csv(OUTPUT_DIR / 'beta_only_paper_roi_diagnostic.csv', index=False)
display(beta_only.groupby('region', sort=False).agg(
    matched=('matched_correlation', 'mean'),
    mismatched=('mismatched_correlation', 'mean'),
    target_specific=('target_specific_contrast', 'mean'),
    subjects=('subject', 'nunique'),
))

subj01
subj02
subj03
subj04
subj05
subj06
subj07
subj08


,matched,mismatched,target_specific,subjects
region,,,,
early_visual,0.082309,-0.016533,0.098843,8
higher_visual,0.071873,-0.014375,0.086248,8
visual_cortex,0.079090,-0.015880,0.094971,8


## 2. Exact Table 1 score for reconstructed images

The repository script `scripts/score_table1_brain_correlation.py` implements the exact scoring path: measured betas → run-wise normalization → repeat average → GNet prediction from each reconstruction → Pearson correlation across ROI voxels.

Expected reconstruction shape is `targets × samples × channels × height × width` (channels-last also works). A MindEye output containing all 18 targets is automatically reduced to sets A and B. Run one subject and task at a time, then aggregate the subject summaries below.

In [9]:
# Edit these paths after generating or downloading reconstruction samples.
RUN_EXACT_SCORE = False
SUBJECT = 1
TASK = 'imagery'
METHOD = 'MindEye2'
RECONSTRUCTIONS = Path('/path/to/all_recons_imagery.pt')
GNET_CHECKPOINT = Path('/path/to/gnet_multisubject.pt')
BRAIN_REGION_MASKS = Path('/path/to/brain_region_masks.hdf5')

prefix = OUTPUT_DIR / f'{METHOD}_subj{SUBJECT:02d}_{TASK}'
command = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'score_table1_brain_correlation.py'),
    '--data-root', str(DATA_ROOT),
    '--subject', str(SUBJECT),
    '--task', TASK,
    '--reconstructions', str(RECONSTRUCTIONS),
    '--gnet-checkpoint', str(GNET_CHECKPOINT),
    '--method', METHOD,
    '--brain-region-masks', str(BRAIN_REGION_MASKS),
    '--predicted-cache', str(prefix) + '_gnet_betas.npy',
    '--output-prefix', str(prefix),
]
print(' '.join(command))
if RUN_EXACT_SCORE:
    subprocess.run(command, check=True)

/srv/conda/envs/nsdimagery/bin/python /home/jovyan/NHprojectNSDimagery/scripts/score_table1_brain_correlation.py --data-root /home/jovyan/NHprojectNSDimagery/data/nsd --subject 1 --task imagery --reconstructions /path/to/all_recons_imagery.pt --gnet-checkpoint /path/to/gnet_multisubject.pt --method MindEye2 --brain-region-masks /path/to/brain_region_masks.hdf5 --predicted-cache /home/jovyan/NHprojectNSDimagery/outputs/05_paper_brain_correlation/MindEye2_subj01_imagery_gnet_betas.npy --output-prefix /home/jovyan/NHprojectNSDimagery/outputs/05_paper_brain_correlation/MindEye2_subj01_imagery


## 3. Aggregate subjects

The paper's reconstruction comparison uses subjects 1, 2, 5, and 7 because those subjects have full 40-session NSD-trained decoders. Run the scoring cell for both tasks and all four subjects. Equal numbers of targets and samples make the mean of subject means equal to the pooled mean.

In [10]:
summary_files = sorted(OUTPUT_DIR.glob('*_summary.csv'))
if summary_files:
    subject_summaries = pd.concat(
        [pd.read_csv(path) for path in summary_files], ignore_index=True
    )
    group_table = (
        subject_summaries.groupby(['method', 'task', 'region'], sort=False)
        .agg(
            brain_correlation=('brain_correlation', 'mean'),
            sd_across_subjects=('brain_correlation', 'std'),
            n_subjects=('subject', 'nunique'),
        )
        .reset_index()
    )
    display(group_table)
else:
    print('No exact-score summaries yet; set RUN_EXACT_SCORE=True after configuring files.')

,method,task,region,brain_correlation,sd_across_subjects,n_subjects
0,GroundTruth_GNet,imagery,early_visual,0.083512,0.032730,4
1,GroundTruth_GNet,imagery,higher_visual,0.096628,0.059087,4
2,GroundTruth_GNet,imagery,visual_cortex,0.086674,0.044919,4
3,GroundTruth_GNet,vision,early_visual,0.375763,0.075326,4
4,GroundTruth_GNet,vision,higher_visual,0.215559,0.041794,4
5,GroundTruth_GNet,vision,visual_cortex,0.280533,0.050808,4


### Split by target type

Set A contains the six simple targets (bars, crosses, and related shapes); Set B contains the six real-scene targets. This table is computed from the target-level `*_detail.csv` files. Correlations are first averaged within each subject and target type, so subjects—not target/sample rows—remain the unit used for the group mean and standard deviation.

In [11]:
detail_files = sorted(OUTPUT_DIR.glob('*_detail.csv'))
if detail_files:
    target_scores = pd.concat(
        [pd.read_csv(path) for path in detail_files], ignore_index=True
    )
    target_scores = target_scores[
        target_scores['stimulus_set'].isin(['A', 'B'])
    ].copy()
    target_scores['target_type'] = target_scores['stimulus_set'].map({
        'A': 'simple',
        'B': 'real_scene',
    })

    subject_target_type = (
        target_scores.groupby(
            ['method', 'subject', 'task', 'target_type', 'region'],
            sort=False,
        )
        .agg(
            brain_correlation=('brain_correlation', 'mean'),
            n_target_samples=('brain_correlation', 'size'),
        )
        .reset_index()
    )
    target_type_table = (
        subject_target_type.groupby(
            ['method', 'task', 'target_type', 'region'], sort=False
        )
        .agg(
            brain_correlation=('brain_correlation', 'mean'),
            sd_across_subjects=('brain_correlation', 'std'),
            n_subjects=('subject', 'nunique'),
        )
        .reset_index()
    )
    target_type_table.to_csv(
        OUTPUT_DIR / 'group_table_by_target_type.csv', index=False
    )
    display(target_type_table)
else:
    print('No target-level detail files yet; run the exact scoring cell first.')

,method,task,target_type,region,brain_correlation,sd_across_subjects,n_subjects
0,GroundTruth_GNet,imagery,simple,early_visual,0.085290,0.046031,4
1,GroundTruth_GNet,imagery,real_scene,early_visual,0.081734,0.029655,4
2,GroundTruth_GNet,imagery,simple,higher_visual,0.050895,0.041586,4
3,GroundTruth_GNet,imagery,real_scene,higher_visual,0.142360,0.080135,4
4,GroundTruth_GNet,imagery,simple,visual_cortex,0.054388,0.039078,4
5,GroundTruth_GNet,imagery,real_scene,visual_cortex,0.118960,0.052516,4
6,GroundTruth_GNet,vision,simple,early_visual,0.294622,0.085584,4
7,GroundTruth_GNet,vision,real_scene,early_visual,0.456904,0.073549,4
8,GroundTruth_GNet,vision,simple,higher_visual,0.101475,0.029349,4
9,GroundTruth_GNet,vision,real_scene,higher_visual,0.329643,0.055552,4


## Interpretation guardrails

- The main paper claim concerns the **vision-to-imagery drop** by ROI, not simply whether the imagery early-visual score is numerically below the higher-visual score. Compare each method's imagery result with its own vision result.
- The beta-only target-specific contrast asks whether target identity is shared across conditions. It cannot validate a reconstruction model.
- Full Table 1 reproduction requires the same reconstruction model/checkpoint, ten random samples, four subjects, and the authors' seed. With another seed, expect close rather than bit-identical values.